# 簡單機器學習實例

> 📚 **目標**：通過實際例子學習機器學習的基本應用
>
> ⏱️ **時長**：60-90 分鐘
>
> 💡 **前置要求**：完成 `01_ml_concepts_demo.ipynb`

## 本 Notebook 內容

1. **房價預測（回歸）** - 預測連續值
2. **鳶尾花分類（分類）** - 預測離散類別
3. **手寫數字識別預覽** - 深度學習應用
4. **🤖 AI 輔助實踐** - 使用 AI 工具優化代碼

In [ ]:
# 導入所有必要的庫
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix
from sklearn.datasets import load_iris, load_digits
import warnings
warnings.filterwarnings('ignore')

# 設置繪圖風格
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✓ 環境準備完成！")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 1. 房價預測（回歸任務）

### 任務描述
- **目標**：根據房屋特徵預測房價
- **類型**：監督學習 - 回歸
- **算法**：線性回歸

### 步驟
1. 生成模擬數據
2. 探索性數據分析
3. 訓練模型
4. 評估性能
5. 進行預測

In [ ]:
# 1.1 生成模擬房價數據
np.random.seed(42)

# 特徵：面積、房間數、樓層、建造年份
n_samples = 200
area = np.random.uniform(50, 200, n_samples)  # 平方米
rooms = np.random.randint(1, 6, n_samples)      # 房間數
floor = np.random.randint(1, 21, n_samples)     # 樓層
year_built = np.random.randint(1990, 2024, n_samples)  # 建造年份

# 生成價格（基於特徵的線性組合 + 噪音）
price = (area * 0.5 +      # 面積影響
         rooms * 10 +       # 房間數影響  
         floor * 2 +        # 樓層影響
         (year_built - 1990) * 1.5 +  # 年份影響
         np.random.normal(0, 10, n_samples))  # 噪音

# 創建 DataFrame
df_house = pd.DataFrame({
    'area': area,
    'rooms': rooms,
    'floor': floor,
    'year_built': year_built,
    'price': price
})

print("數據集概覽：")
print(df_house.head())
print(f"\n數據形狀: {df_house.shape}")
print("\n統計信息:")
print(df_house.describe())

In [ ]:
# 1.2 探索性數據分析（EDA）
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 面積 vs 價格
axes[0, 0].scatter(df_house['area'], df_house['price'], alpha=0.6)
axes[0, 0].set_xlabel('Area (sqm)')
axes[0, 0].set_ylabel('Price')
axes[0, 0].set_title('Area vs Price')

# 房間數 vs 價格
axes[0, 1].scatter(df_house['rooms'], df_house['price'], alpha=0.6)
axes[0, 1].set_xlabel('Number of Rooms')
axes[0, 1].set_ylabel('Price')
axes[0, 1].set_title('Rooms vs Price')

# 樓層 vs 價格
axes[1, 0].scatter(df_house['floor'], df_house['price'], alpha=0.6)
axes[1, 0].set_xlabel('Floor')
axes[1, 0].set_ylabel('Price')
axes[1, 0].set_title('Floor vs Price')

# 建造年份 vs 價格
axes[1, 1].scatter(df_house['year_built'], df_house['price'], alpha=0.6)
axes[1, 1].set_xlabel('Year Built')
axes[1, 1].set_ylabel('Price')
axes[1, 1].set_title('Year Built vs Price')

plt.tight_layout()
plt.show()

# 相關性矩陣
plt.figure(figsize=(10, 8))
sns.heatmap(df_house.corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.show()

print("\n📊 觀察：")
print("- 面積與價格呈現強正相關")
print("- 房間數、樓層、建造年份也影響價格")
print("- 可以用線性模型來預測價格")

In [ ]:
# 1.3 準備數據並訓練模型
# 分離特徵和目標
X = df_house[['area', 'rooms', 'floor', 'year_built']]
y = df_house['price']

# 分割訓練集和測試集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"訓練集大小: {X_train.shape[0]} 樣本")
print(f"測試集大小: {X_test.shape[0]} 樣本")

# 創建並訓練模型
model_regression = LinearRegression()
model_regression.fit(X_train, y_train)

print("\n✓ 模型訓練完成！")
print("\n學習到的參數：")
for feature, coef in zip(X.columns, model_regression.coef_):
    print(f"  {feature}: {coef:.4f}")
print(f"  截距: {model_regression.intercept_:.4f}")

In [ ]:
# 1.4 評估模型性能
# 在訓練集和測試集上進行預測
y_train_pred = model_regression.predict(X_train)
y_test_pred = model_regression.predict(X_test)

# 計算評估指標
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("📊 模型性能評估：")
print(f"\n訓練集：")
print(f"  均方誤差 (MSE): {train_mse:.2f}")
print(f"  R² 分數: {train_r2:.4f}")
print(f"\n測試集：")
print(f"  均方誤差 (MSE): {test_mse:.2f}")
print(f"  R² 分數: {test_r2:.4f}")

print("\n💡 解釋：")
print(f"  R² = {test_r2:.2%} 表示模型解釋了 {test_r2:.2%} 的價格變異")

# 可視化預測結果
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_test, y_test_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
         'r--', linewidth=2, label='Perfect prediction')
plt.xlabel('True Price')
plt.ylabel('Predicted Price')
plt.title('Predictions vs True Values')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
residuals = y_test - y_test_pred
plt.scatter(y_test_pred, residuals, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
plt.xlabel('Predicted Price')
plt.ylabel('Residuals')
plt.title('Residual Plot')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 1.5 使用模型進行新預測
# 創建新房屋的特徵
new_houses = pd.DataFrame({
    'area': [100, 150, 80],
    'rooms': [3, 4, 2],
    'floor': [5, 10, 3],
    'year_built': [2020, 2015, 2023]
})

# 預測價格
predicted_prices = model_regression.predict(new_houses)

print("\n🏠 新房屋價格預測：")
print("="*60)
for i, (idx, row) in enumerate(new_houses.iterrows()):
    print(f"\n房屋 {i+1}:")
    print(f"  面積: {row['area']:.0f} 平方米")
    print(f"  房間數: {row['rooms']}")
    print(f"  樓層: {row['floor']}")
    print(f"  建造年份: {row['year_built']}")
    print(f"  預測價格: {predicted_prices[i]:.2f} 萬元")

print("\n💡 試試看：修改上面的特徵值，觀察預測如何變化！")

## 2. 鳶尾花分類（分類任務）

### 任務描述
- **目標**：根據花朵特徵分類鳶尾花品種
- **類型**：監督學習 - 多分類
- **數據集**：經典的 Iris 數據集
- **算法**：決策樹分類器

### 品種
- Setosa（山鳶尾）
- Versicolor（雜色鳶尾）
- Virginica（維吉尼亞鳶尾）

In [ ]:
# 2.1 加載 Iris 數據集
iris = load_iris()
X_iris = iris.data
y_iris = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

# 創建 DataFrame
df_iris = pd.DataFrame(X_iris, columns=feature_names)
df_iris['species'] = y_iris
df_iris['species_name'] = df_iris['species'].map(
    {0: target_names[0], 1: target_names[1], 2: target_names[2]}
)

print("📊 鳶尾花數據集概覽：")
print(df_iris.head())
print(f"\n數據形狀: {df_iris.shape}")
print(f"\n特徵: {feature_names}")
print(f"類別: {list(target_names)}")
print(f"\n每個類別的樣本數:")
print(df_iris['species_name'].value_counts())

In [ ]:
# 2.2 數據可視化
# 使用前兩個特徵進行可視化
plt.figure(figsize=(15, 5))

# 散點圖
plt.subplot(1, 3, 1)
for i, species in enumerate(target_names):
    mask = df_iris['species'] == i
    plt.scatter(df_iris[mask]['sepal length (cm)'], 
                df_iris[mask]['sepal width (cm)'],
                label=species, alpha=0.6, s=50)
plt.xlabel('Sepal Length (cm)')
plt.ylabel('Sepal Width (cm)')
plt.title('Sepal Dimensions')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
for i, species in enumerate(target_names):
    mask = df_iris['species'] == i
    plt.scatter(df_iris[mask]['petal length (cm)'], 
                df_iris[mask]['petal width (cm)'],
                label=species, alpha=0.6, s=50)
plt.xlabel('Petal Length (cm)')
plt.ylabel('Petal Width (cm)')
plt.title('Petal Dimensions')
plt.legend()
plt.grid(True, alpha=0.3)

# 箱線圖
plt.subplot(1, 3, 3)
df_iris_plot = df_iris.melt(id_vars=['species_name'], 
                             value_vars=feature_names,
                             var_name='feature', 
                             value_name='value')
sns.boxplot(data=df_iris_plot, x='feature', y='value', hue='species_name')
plt.xticks(rotation=45, ha='right')
plt.title('Feature Distributions')

plt.tight_layout()
plt.show()

print("\n📊 觀察：")
print("- Setosa 在花瓣尺寸上明顯與其他兩種不同")
print("- Versicolor 和 Virginica 有一定重疊，但仍可區分")
print("- 花瓣特徵比萼片特徵更具區分性")

In [ ]:
# 2.3 訓練分類模型
# 分割數據
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)

print(f"訓練集大小: {X_train_iris.shape[0]} 樣本")
print(f"測試集大小: {X_test_iris.shape[0]} 樣本")

# 創建並訓練決策樹分類器
clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X_train_iris, y_train_iris)

print("\n✓ 分類器訓練完成！")
print(f"決策樹深度: {clf.get_depth()}")
print(f"葉節點數: {clf.get_n_leaves()}")

In [ ]:
# 2.4 評估分類性能
# 預測
y_train_pred_iris = clf.predict(X_train_iris)
y_test_pred_iris = clf.predict(X_test_iris)

# 計算準確率
train_acc = accuracy_score(y_train_iris, y_train_pred_iris)
test_acc = accuracy_score(y_test_iris, y_test_pred_iris)

print("📊 模型性能評估：")
print(f"\n訓練集準確率: {train_acc:.2%}")
print(f"測試集準確率: {test_acc:.2%}")

print("\n詳細分類報告（測試集）：")
print(classification_report(y_test_iris, y_test_pred_iris, 
                          target_names=target_names))

# 混淆矩陣
cm = confusion_matrix(y_test_iris, y_test_pred_iris)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names,
            yticklabels=target_names)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

print("\n💡 混淆矩陣解釋：")
print("- 對角線：正確分類的樣本數")
print("- 非對角線：錯誤分類的樣本數")

In [ ]:
# 2.5 預測新樣本
# 創建新的鳶尾花樣本
new_flowers = np.array([
    [5.1, 3.5, 1.4, 0.2],  # 看起來像 Setosa
    [6.2, 2.9, 4.3, 1.3],  # 看起來像 Versicolor
    [7.2, 3.0, 5.8, 1.6],  # 看起來像 Virginica
])

# 預測
predictions = clf.predict(new_flowers)
probabilities = clf.predict_proba(new_flowers)

print("\n🌸 新鳶尾花樣本預測：")
print("="*70)
for i, (sample, pred, prob) in enumerate(zip(new_flowers, predictions, probabilities)):
    print(f"\n樣本 {i+1}:")
    print(f"  特徵: [萼片長={sample[0]}, 萼片寬={sample[1]}, "
          f"花瓣長={sample[2]}, 花瓣寬={sample[3]}]")
    print(f"  預測類別: {target_names[pred]}")
    print(f"  置信度:")
    for j, species in enumerate(target_names):
        print(f"    {species}: {prob[j]:.2%}")

print("\n💡 試試看：修改特徵值，觀察預測如何變化！")

## 3. 手寫數字識別預覽

### 簡單的圖像分類

這是深度學習的經典入門問題，我們這裡用簡單方法先體驗一下。

In [ ]:
# 3.1 加載手寫數字數據集
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print(f"數據集大小: {X_digits.shape[0]} 個圖像")
print(f"每個圖像: {digits.images[0].shape} = {X_digits.shape[1]} 像素")
print(f"類別數: {len(np.unique(y_digits))} (數字 0-9)")

# 可視化一些示例
fig, axes = plt.subplots(2, 5, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray')
    ax.set_title(f'Label: {digits.target[i]}')
    ax.axis('off')
plt.suptitle('Sample Handwritten Digits', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# 3.2 快速訓練和評估
X_train_dig, X_test_dig, y_train_dig, y_test_dig = train_test_split(
    X_digits, y_digits, test_size=0.2, random_state=42
)

# 使用邏輯回歸
clf_digits = LogisticRegression(max_iter=1000, random_state=42)
clf_digits.fit(X_train_dig, y_train_dig)

# 評估
y_pred_dig = clf_digits.predict(X_test_dig)
accuracy = accuracy_score(y_test_dig, y_pred_dig)

print(f"測試集準確率: {accuracy:.2%}")

# 顯示一些預測結果
fig, axes = plt.subplots(2, 5, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    idx = i
    image = X_test_dig[idx].reshape(8, 8)
    true_label = y_test_dig[idx]
    pred_label = y_pred_dig[idx]
    
    ax.imshow(image, cmap='gray')
    color = 'green' if true_label == pred_label else 'red'
    ax.set_title(f'True: {true_label}, Pred: {pred_label}', color=color)
    ax.axis('off')
    
plt.suptitle('Predictions (Green=Correct, Red=Wrong)', fontsize=16)
plt.tight_layout()
plt.show()

print("\n💡 這只是簡單演示！")
print("在後續章節中，我們會用深度學習（卷積神經網絡）達到更高的準確率！")

## 4. 🤖 AI 輔助學習實踐

### 使用 AI 學習助手

在學習過程中，你可以使用我們提供的 AI 工具來：
1. 理解困難的概念
2. 生成更多代碼示例
3. 獲取個性化學習建議

In [ ]:
# 示例：如何使用 AI 學習助手
print("🤖 AI 學習助手使用示例\n")
print("在命令行中運行以下命令：\n")

print("1. 解釋概念：")
print("   python ai_learning_assistant.py --mode explain --concept 監督學習\n")

print("2. 生成代碼示例：")
print("   python ai_learning_assistant.py --mode code --task 分類\n")

print("3. 查看學習路徑：")
print("   python ai_learning_assistant.py --mode roadmap --level beginner\n")

print("4. 交互式模式：")
print("   python ai_learning_assistant.py --interactive\n")

print("5. 進行測驗：")
print("   python quiz_generator.py --interactive")

## 💡 練習題

鞏固你的學習，嘗試完成以下練習：

### 練習 1：改進房價預測
1. 添加更多特徵（如距離市中心、是否有電梯等）
2. 嘗試不同的模型（如決策樹、隨機森林）
3. 使用交叉驗證評估模型

### 練習 2：擴展鳶尾花分類
1. 只使用部分特徵，觀察性能變化
2. 比較不同分類器（邏輯回歸、SVM、隨機森林）
3. 繪製決策邊界

### 練習 3：探索更多數據集
1. 使用 scikit-learn 的其他內置數據集
2. 從 Kaggle 下載真實數據集
3. 應用本 Notebook 學到的技術

### 提示
- 使用 `ai_learning_assistant.py` 獲取幫助
- 參考 scikit-learn 官方文檔
- 嘗試不同的超參數

In [ ]:
# 你的練習代碼區域
# 在這裡嘗試上面的練習！

# 練習 1: 房價預測改進
# TODO: 你的代碼

# 練習 2: 鳶尾花分類擴展
# TODO: 你的代碼

# 練習 3: 探索新數據集
# TODO: 你的代碼

print("開始你的練習吧！")

## 總結

### 🎯 你學到了什麼

通過本 Notebook，你已經：

1. **掌握了回歸任務**
   - 預測連續值（房價）
   - 評估指標：MSE、R²
   - 解釋模型參數

2. **掌握了分類任務**
   - 預測離散類別（花的種類）
   - 評估指標：準確率、混淆矩陣
   - 理解決策過程

3. **體驗了圖像分類**
   - 處理高維數據
   - 為深度學習打下基礎

4. **學會使用工具**
   - scikit-learn 的基本工作流
   - 數據預處理和可視化
   - AI 輔助學習工具

### ➡️ 下一步

1. 完成練習題
2. 使用 `quiz_generator.py` 測試理解
3. 準備進入第 2 章學習數學基礎
4. 探索 scikit-learn 的更多功能

### 📚 推薦資源

- [Scikit-learn 官方教程](https://scikit-learn.org/stable/tutorial/)
- [Kaggle Learn](https://www.kaggle.com/learn) - 實踐導向的課程
- [機器學習速成課程](https://developers.google.com/machine-learning/crash-course)

---

**恭喜完成實踐！繼續加油！🚀**